<a href="https://colab.research.google.com/github/srj1407/Practice/blob/main/Colab/From_APIs_To_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Todos

**🏋️ Todo 1: Context Management**
Our agent re-sends everything each iteration. Add summarization: after N iterations, compress the conversation history. Compare token usage before/after.

**🏋️ Todo 2: Planning Step**
Modify the agent to first create a PLAN (list of steps), then execute each one. Compare Plan→Execute vs pure ReAct on the same task.

**🏋️ Todo 3: Multi-Agent**
Create a 'researcher' (Wikipedia tools) and a 'coder' (file/code tools). Build an orchestrator that routes subtasks to the right agent.

**🏋️ Todo 4: Web Browsing**
Add a `fetch_webpage(url)` tool. The agent can now follow links. What new failure modes appear?



In [ ]:
!pip install openai requests -q
!pip install tavily-python

In [ ]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata
from tavily import TavilyClient
from IPython.display import display, Markdown
from pprint import pprint

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["TAVILY"] = userdata.get("TAVILY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
# client = OpenAI(
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
#     api_key=os.environ["GEMINI_API_KEY"],
# )
tavily_client = TavilyClient(api_key=os.environ["TAVILY"])
TOOL_MODEL = "deepseek/deepseek-v3.2"
# TOOL_MODEL = "stepfun/step-3.5-flash:free"
# TOOL_MODEL = "gemini-2.5-flash"

# TODO 1

In [ ]:
messages = [
        {"role": "system", "content":
            "You are a helpful research assistant."},
        {"role": "user", "content": "Hi"},
    ]
response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            temperature=0, max_tokens=800,
        )
print(response)

In [ ]:
NATIVE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '(5+3)*2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get the current date and time.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the web with Tavily for up-to-date information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
]

def calculate_math(expression):
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() eE")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": f"Invalid characters in: {expression}"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_current_date():
    """Get the current date and time."""
    from datetime import datetime
    return json.dumps({"datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

def tavily_search(**kwargs):
    # Pass ALL supported kwargs straight to Tavily
    results = tavily_client.search(**kwargs)
    return json.dumps(results)

NATIVE_FNS = {
    "tavily_search": tavily_search,
    "calculate": calculate_math,
    "get_current_date": get_current_date,
}

token_log = []


def run_native_agent(user_query, max_iterations=10, verbose=True):
    """
    Agent using native function calling.
    No regex — the API returns structured tool_calls.
    """
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. Use tools to find information. "
            "Think step by step. After gathering enough info, give a complete answer."},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 USER: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} ---")

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=NATIVE_TOOLS, temperature=0, max_tokens=800,
        )

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        # Text response = done
        if finish == "stop" and msg.content:
            if verbose:
                print(f"✅ FINAL ANSWER:\n  {msg.content[:500]}")
            return {"answer": msg.content, "iterations": i + 1, "token_log": token_log}

        # Tool calls
        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)[:80]})")

                if fn_name in NATIVE_FNS:
                    result = NATIVE_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result[:150]}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break

    return {"answer": "Max iterations reached", "iterations": max_iterations, "token_log": token_log}

In [ ]:
result = run_native_agent(
    "What's the result when 21 is added to the result of 100 multiplied by the days returned from the difference in the birth dates of Prime Minister of India and LoP of India. "
)

In [ ]:
print("\n" + "=" * 60)
print("TOKEN USAGE PER ITERATION")
print("=" * 60)
total = 0
for t in token_log:
    subtotal = t['in'] + t['out']
    total += subtotal
    bar = "█" * (t['in'] // 50)
    print(f"  Iter {t['iter']}: {t['in']:>5} in + {t['out']:>4} out = {subtotal:>5} │ {bar}")

print(f"\n  TOTAL: {total:,} tokens")
print(f"  → prompt_tokens GROWS each iteration — the context engineering problem.")

In [ ]:
NATIVE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '(5+3)*2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get the current date and time.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the web with Tavily for up-to-date information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
]

def calculate_math(expression):
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() eE")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": f"Invalid characters in: {expression}"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_current_date():
    """Get the current date and time."""
    from datetime import datetime
    return json.dumps({"datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

def tavily_search(**kwargs):
    # Pass ALL supported kwargs straight to Tavily
    results = tavily_client.search(**kwargs)
    return json.dumps(results)

NATIVE_FNS = {
    "tavily_search": tavily_search,
    "calculate": calculate_math,
    "get_current_date": get_current_date,
}

token_log = []


def run_native_agent(user_query, max_iterations=15, verbose=True):
    """
    Agent using native function calling.
    No regex — the API returns structured tool_calls.
    """
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. Use tools to find information. "
            "Think step by step. After gathering enough info, give a complete answer."},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 USER: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} ---")

        if i!=0 and i%3==0:
          summarizer_message = [
              {"role": "system", "content":
                  "You are a helpful research assistant. You are given a list of messages of conversation between AI and user. "
                  "You have to summarize and compress all the messages and return a single message containing all the relevant information till now."
                  "The message returned should be to the point and accurate so that this can be passed to the llm for taking the next steps for solving the user query."},
              {"role": "user", "content": f"""Return the summary for the following conversation:
              {messages}"""}
          ]
          if verbose:
            print(f"\n--- Summarising messages till now.. ---")
          response = client.chat.completions.create(
            model=TOOL_MODEL, messages=summarizer_message,
            tools=NATIVE_TOOLS, temperature=0, max_tokens=800,
          )
          print(response.choices[0].message.content)

          tokens_in = response.usage.prompt_tokens
          tokens_out = response.usage.completion_tokens
          token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

          messages = [
            {"role": "system", "content":
                f"""You are a helpful research assistant. You use tools to find information.
                You think step by step. After gathering enough info, you give a complete answer.
                You are being given a summary of conversation till now.
                You take the given info and proceed with the next steps.

                Conversation Summary:
                {messages}

                """},
            {"role": "user", "content": user_query},
         ]

          continue

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=NATIVE_TOOLS, temperature=0, max_tokens=800,
        )

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        # Text response = done
        if finish == "stop" and msg.content:
            if verbose:
                print(f"✅ FINAL ANSWER:\n  {msg.content[:500]}")
            return {"answer": msg.content, "iterations": i + 1, "token_log": token_log}

        # Tool calls
        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)[:80]})")

                if fn_name in NATIVE_FNS:
                    result = NATIVE_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result[:150]}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break

    return {"answer": "Max iterations reached", "iterations": max_iterations, "token_log": token_log}

In [ ]:
result = run_native_agent(
    "What's the result when 21 is added to the result of 100 multiplied by the days returned from the difference in the birth dates of Prime Minister of India and LoP of India. "
)

In [ ]:
print("\n" + "=" * 60)
print("TOKEN USAGE PER ITERATION")
print("=" * 60)
total = 0
for t in token_log:
    subtotal = t['in'] + t['out']
    total += subtotal
    bar = "█" * (t['in'] // 50)
    print(f"  Iter {t['iter']}: {t['in']:>5} in + {t['out']:>4} out = {subtotal:>5} │ {bar}")

print(f"\n  TOTAL: {total:,} tokens")
print(f"  → prompt_tokens GROWS each iteration — the context engineering problem.")

# TODO 2

In [ ]:
NATIVE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '(5+3)*2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get the current date and time.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the web with Tavily for up-to-date information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "The path of the file to be read"},
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write something to a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "The path of the file to write something to"},
                    "content": {"type": "string", "description": "The content to write."},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_python",
            "description": "Execute Python code and return stdout/stderr.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string", "description": "The code to run"},
                },
                "required": ["code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files in a directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "The path for which files are to be listed."},
                },
            },
        },
    },
]

def calculate_math(expression):
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() eE")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": f"Invalid characters in: {expression}"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_current_date():
    """Get the current date and time."""
    from datetime import datetime
    return json.dumps({"datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

def tavily_search(**kwargs):
    # Pass ALL supported kwargs straight to Tavily
    results = tavily_client.search(**kwargs)
    return json.dumps(results)

def read_file(path):
    """Read a file's contents."""
    try:
        with open(path, "r") as f:
            content = f.read()
        return json.dumps({"path": path, "content": content[:3000], "truncated": len(content) > 3000})
    except Exception as e:
        return json.dumps({"error": str(e)})

def write_file(path, content):
    """Write content to a file."""
    try:
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return json.dumps({"status": "success", "path": path, "bytes": len(content)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def run_python(code):
    """Execute Python code and return stdout/stderr."""
    try:
        result = subprocess.run(
            ["python3", "-c", code],
            capture_output=True, text=True, timeout=15,
        )
        return json.dumps({
            "stdout": result.stdout[:2000] if result.stdout else "",
            "stderr": result.stderr[:2000] if result.stderr else "",
            "exit_code": result.returncode,
        })
    except subprocess.TimeoutExpired:
        return json.dumps({"error": "Timed out (15s limit)"})
    except Exception as e:
        return json.dumps({"error": str(e)})

def list_files(path="."):
    """List files in a directory."""
    try:
        items = sorted(os.listdir(path))
        return json.dumps({"path": path, "files": items[:50]})
    except Exception as e:
        return json.dumps({"error": str(e)})

NATIVE_FNS = {
    "tavily_search": tavily_search,
    "calculate": calculate_math,
    "get_current_date": get_current_date,
    "read_file": read_file,
    "write_file": write_file,
    "run_python": run_python,
    "list_files": list_files,
}

token_log = []


def run_native_agent(user_query, max_iterations=10, verbose=True):
    """
    Agent using native function calling.
    No regex — the API returns structured tool_calls.
    """

    prompt = """"You are a helpful research assistant. Use tools to find information. You think and first list down the steps required to reach to the final answer.
After making the list, you execute list step by step. You execute only one step at a time.
After executing a step, you think and observe if the result of that step aligns with your final goal.

## How to respond

FIRST MAKE A LIST OF STEPS:
First you make the list giving serial number to each step. You give the list EXACTLY in this format. You must only give steps in the first iteration.
Execution for the steps happen from next iteration.

  <LIST_OF_STEPS>:
  <STEP 1>: ...
  <STEP 2>: ...

Then at each iteration you respond with the following:

  <STEP (Step_No) DONE>
  <RESULTS OF THAT STEP>

If you have reached the final answer you respond like this:

  <FINAL_ANSWER_REACHED>
  <FINAL RESULTS>

"""

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 USER: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} ---")

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=NATIVE_TOOLS, temperature=0, max_tokens=800,
        )

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)[:80]})")

                if fn_name in NATIVE_FNS:
                    result = NATIVE_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result[:150]}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
              if finish == "stop" and 'FINAL_ANSWER_REACHED' in msg.content:
                if verbose:
                  print(f"✅ FINAL ANSWER:\n  {msg.content[:500]}")
                return {"answer": msg.content, "iterations": i + 1}
              if verbose:
                print(f"✅ CONTENT:\n  {msg.content}")
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break

    return {"answer": "Max iterations reached", "iterations": max_iterations, "token_log": token_log}

In [ ]:
result = run_native_agent(
    "Calculate the sum of the square root of the number of years between the discovery of America and the invention of the printing press, and the current population of the capital city of Canada and write the result to a file."
)

In [ ]:
print("\n" + "=" * 60)
print("TOKEN USAGE PER ITERATION")
print("=" * 60)
total = 0
for t in token_log:
    subtotal = t['in'] + t['out']
    total += subtotal
    bar = "█" * (t['in'] // 50)
    print(f"  Iter {t['iter']}: {t['in']:>5} in + {t['out']:>4} out = {subtotal:>5} │ {bar}")

print(f"\n  TOTAL: {total:,} tokens")
print(f"  → prompt_tokens GROWS each iteration — the context engineering problem.")

# TODO 3

In [ ]:
CODER_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '(5+3)*2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "The path of the file to be read"},
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write something to a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "The path of the file to write something to"},
                    "content": {"type": "string", "description": "The content to write."},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_python",
            "description": "Execute Python code and return stdout/stderr.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string", "description": "The code to run"},
                },
                "required": ["code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files in a directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "The path for which files are to be listed."},
                },
            },
        },
    },
]

RESEARCHER_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get the current date and time.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the web with Tavily for up-to-date information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
]

def calculate_math(expression):
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() eE")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": f"Invalid characters in: {expression}"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_current_date():
    """Get the current date and time."""
    from datetime import datetime
    return json.dumps({"datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

def tavily_search(**kwargs):
    # Pass ALL supported kwargs straight to Tavily
    results = tavily_client.search(**kwargs)
    return json.dumps(results)

def read_file(path):
    """Read a file's contents."""
    try:
        with open(path, "r") as f:
            content = f.read()
        return json.dumps({"path": path, "content": content[:3000], "truncated": len(content) > 3000})
    except Exception as e:
        return json.dumps({"error": str(e)})

def write_file(path, content):
    """Write content to a file."""
    try:
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return json.dumps({"status": "success", "path": path, "bytes": len(content)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def run_python(code):
    """Execute Python code and return stdout/stderr."""
    try:
        result = subprocess.run(
            ["python3", "-c", code],
            capture_output=True, text=True, timeout=15,
        )
        return json.dumps({
            "stdout": result.stdout[:2000] if result.stdout else "",
            "stderr": result.stderr[:2000] if result.stderr else "",
            "exit_code": result.returncode,
        })
    except subprocess.TimeoutExpired:
        return json.dumps({"error": "Timed out (15s limit)"})
    except Exception as e:
        return json.dumps({"error": str(e)})

def list_files(path="."):
    """List files in a directory."""
    try:
        items = sorted(os.listdir(path))
        return json.dumps({"path": path, "files": items[:50]})
    except Exception as e:
        return json.dumps({"error": str(e)})

RESEARCHER_FNS = {
    "tavily_search": tavily_search,
    "get_current_date": get_current_date,
}

CODER_FNS = {
    "calculate": calculate_math,
    "read_file": read_file,
    "write_file": write_file,
    "run_python": run_python,
    "list_files": list_files,
}

token_log = []

In [ ]:
def researcher(query, max_iterations=10, verbose=True):
    """
    Agent for doing web search an getting current time and time.
    """

    prompt = """

    You are a helpful assistant. You can do web search and get current date and time. You can use tools to answer the questions.

    """

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 Query from researcher: {query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} from researcher---")

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=RESEARCHER_TOOLS, temperature=0, max_tokens=800,
        )

        print(response)

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)})")

                if fn_name in RESEARCHER_FNS:
                    result = RESEARCHER_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
              if finish == "stop":
                if verbose:
                  print(f"✅ ANSWER:\n  {msg.content}")
                return {"answer": msg.content}
              if verbose:
                print(f"✅ CONTENT:\n  {msg.content}")
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break
    print({"answer": "Max iterations reached", "iterations": max_iterations})
    return {"answer": "Couldn't find what you were looking for."}

In [ ]:
def coder(query, max_iterations=10, verbose=True):
    """
    Agent for creating, listing, reading files, doing calculations and running python code.
    """

    prompt = """

    You are a helpful assistant. You can create, list, read files, do calculations and run python code. You can use tools to answer the questions.

    """

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 Query from coder: {query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} from coder---")

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=CODER_TOOLS, temperature=0, max_tokens=800,
        )

        print(response)

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)})")

                if fn_name in CODER_FNS:
                    result = CODER_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
              if finish == "stop":
                if verbose:
                  print(f"✅ ANSWER:\n  {msg.content}")
                return {"answer": msg.content}
              if verbose:
                print(f"✅ CONTENT:\n  {msg.content}")
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break
    print({"answer": "Max iterations reached", "iterations": max_iterations})
    return {"answer": "Couldn't find what you were looking for."}

In [ ]:
def orchestrator(query, max_iterations=10, verbose=True):
  """
  Agent orchestrating the workflow deciding which agent to delegate tasks to.
  """

  prompt = """

  You are a helpful assistant. You are an main agent and you have access to coder and researcher agents to delegate tasks to.
  If you think any other agent is required, you delegate task to that agent. Also, you check the responses from the other agent if it is correct or not.
  If the solution to the user query is reached, you return the result.

  Info regarding the agents:

  Researcher agent:
    <AGENT_NAME>: researcher
    <DESC>: This agent has access to find current date and do web search. It will take query as string.

  Coder agent:
    <AGENT_NAME>: coder
    <DESC>: This agent has access to read files, write to file, list all the files, run python code, and calculate math expressions. It will take query as string.

  You respond like this:

    If any agent is required:
      <CALL_AGENT>: <Agent name>
      <AGENT_INPUT>: <Input to be given to agent.>
    Else:
      <FINAL_ANSWER_REACHED>: <Final Answer>

  """

  messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": query},
    ]

  if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 Query from orchestrator: {query}")
        print(f"{'='*60}")

  for i in range(max_iterations):

    if verbose:
        print(f"\n--- Iteration {i+1} from Orchestrator---")

    response = client.chat.completions.create(
        model=TOOL_MODEL, messages=messages,
        temperature=0, max_tokens=800,
    )

    print(response)

    tokens_in = response.usage.prompt_tokens
    tokens_out = response.usage.completion_tokens
    token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

    msg = response.choices[0].message
    finish = response.choices[0].finish_reason

    if '<FINAL_ANSWER_REACHED>' in msg.content:
      if verbose:
        print(f"✅ FINAL ANSWER:\n  {msg.content}")
      return

    if '<CALL_AGENT>' in msg.content:
      agent_name_match = re.search(r"<CALL_AGENT>:\s*([^\n]+)", msg.content)
      agent_name = agent_name_match.group(1).strip() if agent_name_match else None
      agent_input_match = re.search(r"<AGENT_INPUT>:\s*(.+)", msg.content, re.DOTALL)
      agent_input = agent_input_match.group(1).strip() if agent_input_match else None

      if verbose:
        print(f"🔧 AGENT: {agent_name}")
        print(f"🔧 INPUT: {agent_input}")

      if agent_name == 'coder':
        result = coder(agent_input)
      elif agent_name == 'researcher':
        result = researcher(agent_input)

        print("Result after agent call:", result)
        print("Type of result:", type(result))

      messages.append({
                    "role": "assistant",
                    "content": result['answer'],
                })


  print({"answer": "Max iterations reached", "iterations": max_iterations})
  return

In [ ]:
# result = orchestrator(
#     "Calculate the sum of the square root of the number of years between the discovery of America and the invention of the printing press, and the current population of the capital city of Canada and write the result to a file."
# )

result = orchestrator(
    "Get latest news and write to a file called news.txt."
)

# TODO 4

In [ ]:
CODER_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '(5+3)*2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "The path of the file to be read"},
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write something to a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "The path of the file to write something to"},
                    "content": {"type": "string", "description": "The content to write."},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_python",
            "description": "Execute Python code and return stdout/stderr.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string", "description": "The code to run"},
                },
                "required": ["code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files in a directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "The path for which files are to be listed."},
                },
            },
        },
    },
]

RESEARCHER_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get the current date and time.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the web with Tavily for up-to-date information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_webpage",
            "description": "Fetches the content of a given URL.",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "The URL to fetch"}
                },
                "required": ["url"],
            },
        },
    },
]

def calculate_math(expression):
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() eE")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": f"Invalid characters in: {expression}"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_current_date():
    """Get the current date and time."""
    from datetime import datetime
    return json.dumps({"datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

def tavily_search(**kwargs):
    # Pass ALL supported kwargs straight to Tavily
    results = tavily_client.search(**kwargs)
    return json.dumps(results)

def read_file(path):
    """Read a file's contents."""
    try:
        with open(path, "r") as f:
            content = f.read()
        return json.dumps({"path": path, "content": content[:3000], "truncated": len(content) > 3000})
    except Exception as e:
        return json.dumps({"error": str(e)})

def write_file(path, content):
    """Write content to a file."""
    try:
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return json.dumps({"status": "success", "path": path, "bytes": len(content)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def run_python(code):
    """Execute Python code and return stdout/stderr."""
    try:
        result = subprocess.run(
            ["python3", "-c", code],
            capture_output=True, text=True, timeout=15,
        )
        return json.dumps({
            "stdout": result.stdout[:2000] if result.stdout else "",
            "stderr": result.stderr[:2000] if result.stderr else "",
            "exit_code": result.returncode,
        })
    except subprocess.TimeoutExpired:
        return json.dumps({"error": "Timed out (15s limit)"})
    except Exception as e:
        return json.dumps({"error": str(e)})

def list_files(path="."):
    """List files in a directory."""
    try:
        items = sorted(os.listdir(path))
        return json.dumps({"path": path, "files": items[:50]})
    except Exception as e:
        return json.dumps({"error": str(e)})

def fetch_webpage(url):
    """Fetches the content of a given URL."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        return json.dumps({"url": url, "content": response.text[:5000], "truncated": len(response.text) > 5000})
    except requests.exceptions.RequestException as e:
        return json.dumps({"error": str(e), "url": url})

RESEARCHER_FNS = {
    "tavily_search": tavily_search,
    "get_current_date": get_current_date,
    "fetch_webpage": fetch_webpage,
}

CODER_FNS = {
    "calculate": calculate_math,
    "read_file": read_file,
    "write_file": write_file,
    "run_python": run_python,
    "list_files": list_files,
}

token_log = []


In [ ]:
def coder(query, max_iterations=10, verbose=True):
    """
    Agent for creating, listing, reading files, doing calculations and running python code.
    """

    prompt = """

    You are a helpful assistant. You can create, list, read files, do calculations and run python code. You can use tools to answer the questions.

    """

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 Query from coder: {query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} from coder---")

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=CODER_TOOLS, temperature=0, max_tokens=800,
        )

        print(response)

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)})")

                if fn_name in CODER_FNS:
                    result = CODER_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
              if finish == "stop":
                if verbose:
                  print(f"✅ ANSWER:\n  {msg.content}")
                return {"answer": msg.content}
              if verbose:
                print(f"✅ CONTENT:\n  {msg.content}")
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break
    print({"answer": "Max iterations reached", "iterations": max_iterations})
    return {"answer": "Couldn't find what you were looking for."}

In [ ]:
def researcher(query, max_iterations=10, verbose=True):
    """
    Agent for doing web search an getting current time and time.
    """

    prompt = """

    You are a helpful assistant. You can do web search, get current date and time, and fetch webpage content. You can use tools to answer the questions.

    """

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 Query from researcher: {query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} from researcher---")

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=RESEARCHER_TOOLS, temperature=0, max_tokens=800,
        )

        print(response)

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)})")

                if fn_name in RESEARCHER_FNS:
                    result = RESEARCHER_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
              if finish == "stop":
                if verbose:
                  print(f"✅ ANSWER:\n  {msg.content}")
                return {"answer": msg.content}
              if verbose:
                print(f"✅ CONTENT:\n  {msg.content}")
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break
    print({"answer": "Max iterations reached", "iterations": max_iterations})
    return {"answer": "Couldn't find what you were looking for."}

In [ ]:
def orchestrator(query, max_iterations=10, verbose=True):
  """
  Agent orchestrating the workflow deciding which agent to delegate tasks to.
  """

  prompt = """

  You are a helpful assistant. You are an main agent and you have access to coder and researcher agents to delegate tasks to.
  If you think any other agent is required, you delegate task to that agent. Also, you check the responses from the other agent if it is correct or not.
  If the solution to the user query is reached, you return the result.

  Info regarding the agents:

  Researcher agent:
    <AGENT_NAME>: researcher
    <DESC>: This agent has access to find current date, do web search, and fetch webpage content. It will take query as string.

  Coder agent:
    <AGENT_NAME>: coder
    <DESC>: This agent has access to read files, write to file, list all the files, run python code, and calculate math expressions. It will take query as string.

  You respond like this:

    If any agent is required:
      <CALL_AGENT>: <Agent name>
      <AGENT_INPUT>: <Input to be given to agent.>
    Else:
      <FINAL_ANSWER_REACHED>: <Final Answer>

  """

  messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": query},
    ]

  if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 Query from orchestrator: {query}")
        print(f"{'='*60}")

  for i in range(max_iterations):

    if verbose:
        print(f"\n--- Iteration {i+1} from Orchestrator---")

    response = client.chat.completions.create(
        model=TOOL_MODEL, messages=messages,
        temperature=0, max_tokens=800,
    )

    print(response)

    tokens_in = response.usage.prompt_tokens
    tokens_out = response.usage.completion_tokens
    token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

    msg = response.choices[0].message
    finish = response.choices[0].finish_reason

    if '<FINAL_ANSWER_REACHED>' in msg.content:
      if verbose:
        print(f"✅ FINAL ANSWER:\n  {msg.content}")
      return

    if '<CALL_AGENT>' in msg.content:
      agent_name_match = re.search(r"<CALL_AGENT>:\s*([^\n]+)", msg.content)
      agent_name = agent_name_match.group(1).strip() if agent_name_match else None
      agent_input_match = re.search(r"<AGENT_INPUT>:\s*(.+)", msg.content, re.DOTALL)
      agent_input = agent_input_match.group(1).strip() if agent_input_match else None

      if verbose:
        print(f"🔧 AGENT: {agent_name}")
        print(f"🔧 INPUT: {agent_input}")

      if agent_name == 'coder':
        result = coder(agent_input)
      elif agent_name == 'researcher':
        result = researcher(agent_input)

        print("Result after agent call:", result)
        print("Type of result:", type(result))

      messages.append(
                    {
                    "role": "assistant",
                    "content": result['answer'],
                })


  print({"answer": "Max iterations reached", "iterations": max_iterations})
  return

In [ ]:
# result = orchestrator(
#     "Calculate the sum of the square root of the number of years between the discovery of America and the invention of the printing press, and the current population of the capital city of Canada and write the result to a file."
# )

result = orchestrator(
    "Fetch the webpage 'https://www.thehindu.com/' and write contents to a file named webpage.txt."
)